# Stock Return Prediction — AAPL Next-Day Log Return

**Research question:** Can ML models predict the *direction* of next-day returns better than a random walk?

**Design:** Stationary log-return target → 80+ features (technical indicators, rolling stats, calendar) → Ridge, Lasso, Huber, RF, GBM, XGBoost, LightGBM, MLP, Stacking, Weighted Ensemble. Walk-forward CV throughout.

**Key constraint:** Target = next-day **log return** (not price). Raw price gives R²≈0.98 via autocorrelation — that is *not* predictive signal. Direction accuracy > 52% is the exploitability bar.

---
All heavy logic lives in `src/`. This notebook is a thin orchestration layer: load → engineer → model → evaluate → backtest.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", category=UserWarning,  module="lightgbm")
warnings.filterwarnings("ignore", category=FutureWarning, module="yfinance")

import sys, os
# Allow imports from repo root when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ""))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

from src.config import (
    TICKER, START_DATE, END_DATE, SPLIT_DATE, RANDOM_STATE,
    N_CV_SPLITS, CV_GAP,
)
from src.data     import load_price_data, chronological_split
from src.features import build_features, prepare_matrices
from src.evaluate import (
    compute_metrics, print_metrics, select_features,
    walk_forward_cv, print_walk_forward_results, plot_model_comparison,
)
from src.models   import (
    get_linear_models, get_tree_models, get_mlp,
    build_tuned_models, build_voting_ensemble,
    build_stacking_ensemble, build_weighted_ensemble,
)
from src.strategy import run_backtest, print_backtest_summary, plot_backtest

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight", "font.size": 11})
sns.set_palette("husl")
np.random.seed(RANDOM_STATE)

TSCV = TimeSeriesSplit(n_splits=N_CV_SPLITS, gap=CV_GAP)
print("Setup complete.")

## 1. Data Loading

Cached to `data/` on first run to avoid yfinance rate limits.
Multi-level columns (yfinance ≥0.2) are flattened automatically.

In [ ]:
df_raw = load_price_data(TICKER, START_DATE, END_DATE)
print(f"{TICKER} | {df_raw.index.min().date()} → {df_raw.index.max().date()} | {len(df_raw)} rows")
print(f"Price range: ${df_raw['Close'].min():.2f} – ${df_raw['Close'].max():.2f}")
df_raw.tail(3)

## 2. Feature Engineering

Features are built entirely from information available *at close on day t* — no look-ahead.

| Group | Features |
|-------|----------|
| Returns | Log-return lags (1,2,3,5,10d), rolling mean/vol (5,10,20,50d) |
| Momentum | 5/10/20-day price momentum, return acceleration |
| Technicals | RSI, MACD, Bollinger Bands, ATR, ADX, OBV, MFI, CCI, Aroon, Stochastic, Williams %R |
| Volume | Relative volume, 20d z-score, volume-price correlation |
| Structure | 20d support/resistance distance, MA-crossover regime, candlestick proxies |
| Calendar | Day-of-week + month with **sin/cos cyclical encoding** |

In [ ]:
df = build_features(df_raw)
print(f"Feature engineering complete | {df.shape[1]} total columns")

X, y = prepare_matrices(df)
print(f"Clean feature matrix: {X.shape} | Target: mean={y.mean():.5f}, std={y.std():.4f}")

## 3. Train / Test Split

**Split:** Chronological by calendar date (no shuffle — mandatory for time series).  
**Scaling:** `StandardScaler` fit on train only; applied only to linear models and MLP to prevent leakage.

In [ ]:
X_train, X_test, y_train, y_test = chronological_split(X, y, SPLIT_DATE)

print(f"Train: {X_train.index.min().date()} → {X_train.index.max().date()} ({len(X_train)} days)")
print(f"Test : {X_test.index.min().date()}  → {X_test.index.max().date()}  ({len(X_test)} days)")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## 4. Feature Importance & Selection

**Permutation importance** (evaluated on the test set) replaces impurity importance — it is unbiased toward high-cardinality features and directly measures predictive relevance.

Features with negative mean importance are net harmful and are removed before modelling.

In [ ]:
SELECTED, importance_df = select_features(
    X_train.values, y_train.values,
    X_test.values,  y_test.values,
    feature_names=X_train.columns.tolist(),
    plot=True,
)

# Slice to selected features and re-scale for linear models
X_tr = X_train[SELECTED].values
X_te = X_test[SELECTED].values

sc_sel = StandardScaler()
X_tr_sc = sc_sel.fit_transform(X_tr)
X_te_sc = sc_sel.transform(X_te)

## 5. Modelling

All models are trained on the selected feature set.

**Primary metric:** direction accuracy (>50% = better than random; >52% = potentially tradeable).  
**Secondary metrics:** RMSE, MAE, R² (informational — signal-to-noise is very low for daily returns).

In [ ]:
results = []

# Naive baseline: predict 0 (random walk hypothesis)
naive = compute_metrics(y_test, np.zeros(len(y_test)), "Naive (predict 0)")
results.append(naive); print_metrics(naive)
print("  Every model must beat this to add value.\n")

In [ ]:
# Linear models (scaled input)
print("Linear models:")
for name, mdl in get_linear_models().items():
    mdl.fit(X_tr_sc, y_train)
    m = compute_metrics(y_test, mdl.predict(X_te_sc), name)
    results.append(m); print_metrics(m)

In [ ]:
# Tree models (scale-invariant)
print("\nTree models:")
for name, mdl in get_tree_models().items():
    mdl.fit(X_tr, y_train)
    m = compute_metrics(y_test, mdl.predict(X_te), name)
    results.append(m); print_metrics(m)

In [ ]:
# MLP (scaled input)
print("\nNeural network:")
mlp = get_mlp()
mlp.fit(X_tr_sc, y_train)
m = compute_metrics(y_test, mlp.predict(X_te_sc), "MLP")
results.append(m); print_metrics(m)

In [ ]:
# Hyperparameter tuning — RF, XGBoost, LightGBM
print("\nTuning RF, XGBoost, LightGBM (RandomizedSearchCV + TimeSeriesSplit)...")
tuned = build_tuned_models(X_tr, y_train.values, tscv=TSCV, verbose=True)

for name, mdl in tuned.items():
    m = compute_metrics(y_test, mdl.predict(X_te), name)
    results.append(m); print_metrics(m)

In [ ]:
# Ensembles
print("\nEnsembles:")

# 1. Equal-weight voting
voting = build_voting_ensemble(tuned, X_tr, y_train.values)
m = compute_metrics(y_test, voting.predict(X_te), "Voting Ensemble")
results.append(m); print_metrics(m)

# 2. Manual time-series stacking (OOF Ridge meta-learner)
y_stacked = build_stacking_ensemble(
    tuned, X_tr, y_train.values, X_tr_sc, X_te, X_te_sc,
)
m = compute_metrics(y_test, y_stacked, "TSS Stacking")
results.append(m); print_metrics(m)

# 3. Inverse-RMSE weighted average (weights from CV, no test leakage)
y_weighted = build_weighted_ensemble(tuned, X_tr, y_train.values, X_te, TSCV, verbose=True)
m = compute_metrics(y_test, y_weighted, "Weighted Ensemble (OOF)")
results.append(m); print_metrics(m)

## 6. Model Comparison

Walk-forward cross-validation (expanding window) gives a more honest picture than a single train/test split — it simulates production retraining across different market regimes.

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values("Dir Acc (%)", ascending=False)
print("=" * 75)
print("MODEL COMPARISON  (sorted by direction accuracy)")
print("=" * 75)
print(results_df.to_string(index=False, float_format="{:.5f}".format))

best = results_df.iloc[0]
print(f"\nBest model: {best['Model']}  |  Dir Acc: {best['Dir Acc (%)']:.2f}%")

plot_model_comparison(results_df)

In [ ]:
# Walk-forward CV on four representative models
from src.models import get_tree_models
from sklearn.linear_model import Ridge as _Ridge
from xgboost import XGBRegressor as _XGB
from lightgbm import LGBMRegressor as _LGBM
from sklearn.ensemble import GradientBoostingRegressor as _GBM
from src.config import GBM_PARAMS

wf_models = {
    "Ridge":   _Ridge(alpha=10.0, random_state=RANDOM_STATE),
    "XGBoost": tuned["XGBoost (Tuned)"],
    "LGBM":    tuned["LightGBM (Tuned)"],
    "GBM":     _GBM(**GBM_PARAMS),
}
wf_rmse, wf_da = walk_forward_cv(wf_models, X.values, y.values)
print_walk_forward_results(wf_rmse, wf_da)

## 7. Trading Strategy Backtest

Simple long-only strategy: enter when predicted return exceeds a threshold.

- **Signal threshold:** 0.1% predicted log return  
- **Transaction cost:** 0.1% round-trip (applied on every position change)  
- **Metrics:** annualised Sharpe, Sortino, max drawdown, win rate

In [ ]:
result = run_backtest(y_test.values, y_weighted)   # best ensemble
print_backtest_summary(result)
plot_backtest(result, X_test.index)